# ProXtal-LM — Training

Interactive notebook for configuring and launching a training run,
monitoring progress, and visualising training curves.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import torch
import pandas as pd
import matplotlib.pyplot as plt

from scripts.publication_graphs import apply_publication_style
apply_publication_style()

## 1. Configuration

Load a config from a JSON file or use one of the Python presets.

In [ ]:
from scripts.inference import load_config_from_json
from proxtal_lm.config import get_small_config, get_large_config, get_optimized_config

# Option A: load from JSON
# config = load_config_from_json('../configs/large.json')

# Option B: use a Python preset
config = get_large_config()

# Override any settings
config.name = 'notebook_run'
config.training.checkpoint_dir = '../checkpoints/notebook_run'
config.training.max_epochs = 5  # short run for testing

config.__post_init__()
print(config)

## 2. Data Loading

In [ ]:
from proxtal_lm.data import CrystalContactsDataset, collate_pad
from torch.utils.data import DataLoader

train_ds = CrystalContactsDataset(config.data.train_path, config.data.num_bins)
val_ds   = CrystalContactsDataset(config.data.val_path,   config.data.num_bins)

collate_fn = lambda batch: collate_pad(batch, pad_multiple=config.data.pad_multiple)

train_loader = DataLoader(
    train_ds, batch_size=config.data.batch_size, shuffle=True,
    num_workers=config.data.num_workers, collate_fn=collate_fn,
    pin_memory=config.data.pin_memory,
)
val_loader = DataLoader(
    val_ds, batch_size=config.data.batch_size, shuffle=False,
    num_workers=config.data.num_workers, collate_fn=collate_fn,
    pin_memory=config.data.pin_memory,
)

print(f'Train: {len(train_ds)} samples, Val: {len(val_ds)} samples')

## 3. Model Setup

In [ ]:
from proxtal_lm.models import CrystalTriangularModel

device = torch.device(config.device if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

mc = config.model
win = mc.attention_window_size if mc.attention_window_size > 0 else None

model = CrystalTriangularModel(
    emb_dim=mc.emb_dim,
    d_model=mc.d_model,
    d_pair=mc.d_pair,
    n_seq_layers=mc.n_seq_layers,
    n_blocks=mc.n_blocks,
    tri_hidden=mc.tri_hidden,
    out_ch=mc.out_ch,
    use_checkpoint=mc.use_checkpoint,
    n_recycles=mc.n_recycles,
    max_rel_pos=mc.max_rel_pos,
    num_space_groups=mc.num_space_groups,
    n_heads=mc.n_heads,
    attention_window_size=win,
    dropout=mc.dropout,
    n_hypotheses=mc.n_hypotheses,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

## 4. Training Loop

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from proxtal_lm.training import train_one_epoch, validate_one_epoch

opt = torch.optim.AdamW(
    model.parameters(),
    lr=config.training.learning_rate,
    weight_decay=config.training.weight_decay,
)
scaler = torch.amp.GradScaler('cuda') if config.training.use_amp else None

scheduler = None
if config.training.scheduler_type == 'cosine':
    scheduler = CosineAnnealingLR(opt, T_max=config.training.max_epochs)
elif config.training.scheduler_type == 'plateau':
    scheduler = ReduceLROnPlateau(
        opt, mode='min',
        factor=config.training.scheduler_factor,
        patience=config.training.scheduler_patience,
        min_lr=config.training.scheduler_min_lr,
    )

# Tracking
history = {'epoch': [], 'lr': [], 'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(config.training.max_epochs):
    torch.cuda.empty_cache()

    train_metrics = train_one_epoch(
        model, train_loader, opt, device,
        scaler=scaler,
        grad_clip=config.training.grad_clip,
        accum_steps=config.training.accum_steps,
        matching_mode=config.training.matching_mode,
        crystal_og_weight=config.training.crystal_og_weight,
        diversity_weight=config.training.diversity_weight,
        crystal_contact_weight=config.training.crystal_contact_weight,
        contact_threshold=config.training.contact_threshold,
        contact_emphasis=config.training.contact_emphasis,
    )

    val_metrics = validate_one_epoch(model, val_loader, device)
    val_loss = val_metrics.get('val_loss', val_metrics.get('loss'))

    if scheduler:
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

    lr = opt.param_groups[0]['lr']
    history['epoch'].append(epoch + 1)
    history['lr'].append(lr)
    history['train_loss'].append(train_metrics.get('train_loss', 0))
    history['val_loss'].append(val_loss)

    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        ckpt_path = os.path.join(config.training.checkpoint_dir, 'best_checkpoint.pt')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'best_val_loss': best_val_loss,
            'config': config,
        }, ckpt_path)
        improved = ' *'
    else:
        epochs_no_improve += 1

    print(
        f'Epoch {epoch+1:3d}/{config.training.max_epochs} | '
        f'LR {lr:.2e} | '
        f'Train {history["train_loss"][-1]:.4f} | '
        f'Val {val_loss:.4f}{improved}'
    )

    if epochs_no_improve >= config.training.early_stop_patience:
        print(f'Early stopping after {epoch+1} epochs.')
        break

print(f'\nBest validation loss: {best_val_loss:.4f}')

## 5. Training Curves (Live)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(history['epoch'], history['train_loss'], label='Train', color='#0072B2')
ax.plot(history['epoch'], history['val_loss'], label='Val', color='#D55E00')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss')
ax.legend()

ax = axes[1]
ax.plot(history['epoch'], history['lr'], color='#56B4E9')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('LR Schedule')

fig.tight_layout()
plt.show()

## 6. Publication Figures from Saved Metrics

If a `training_metrics.csv` was written (via the CLI training script),
use the publication graphing module to generate polished plots.

In [ ]:
from scripts.publication_graphs import (
    plot_training_curves,
    plot_precision_recall,
    plot_summary_dashboard,
)

CSV_PATH = '../checkpoints/v8_esmc_0/training_metrics.csv'

if os.path.exists(CSV_PATH):
    plot_training_curves(CSV_PATH, mode='view')
    plot_precision_recall(CSV_PATH, mode='view')
    plot_summary_dashboard(CSV_PATH, mode='view')
else:
    print(f'No CSV found at {CSV_PATH}')